# ViT-B/16 - Phan loai anh X-quang phoi tren Kaggle

Notebook nay fine-tune `ViT-B/16` pretrained cho hai lop `NORMAL` va `PNEUMONIA` bang PyTorch tren bo du lieu `dataset_clean_70_15_15`.

Ket qua duoc luu vao `/kaggle/working/outputs/vit_b_16/`, gom checkpoint `.pth`, bieu do training, confusion matrix, ROC curve va anh du doan mau.

**Cach chay tren Kaggle:**
1. Bam `Add Input` va them dataset co chua `dataset_clean_70_15_15`.
2. Bat GPU: `Settings -> Accelerator -> GPU T4/P100`.
3. Bam `Run All`.
4. Sau khi chay xong, bam `Save Version -> Save & Run All` de Kaggle giu output.

**Luu y:** Neu Kaggle khong tai duoc pretrained weights, hay bat Internet trong Settings.


## 1. Cau hinh dataset cho Kaggle

Code ben duoi tu dong tim dataset trong `/kaggle/input`, ho tro ca ten `dataset_clean` va `dataset-clean`. Model va output luu vao `/kaggle/working` de tai ve sau khi train.


In [ ]:
import json
from pathlib import Path

# =========================
# CẤU HÌNH DATASET CHO KAGGLE
# =========================
# Trên Kaggle, bấm Add Input và thêm dataset có chứa thư mục dataset_clean_70_15_15.
# Cấu trúc thường gặp:
# /kaggle/input/dataset_clean/dataset/dataset_clean_70_15_15/train
# /kaggle/input/dataset_clean/dataset/dataset_clean_70_15_15/val
# /kaggle/input/dataset_clean/dataset/dataset_clean_70_15_15/test
#
# Kaggle đôi khi đổi dấu "_" thành "-", nên code sẽ tự tìm trong toàn bộ /kaggle/input.

SPLIT_SEED = 42
CLASSES = ('NORMAL', 'PNEUMONIA')
DATASET_FOLDER_NAME = 'dataset_clean_70_15_15'


def prepared_dataset_is_complete(data_dir):
    """Kiểm tra dataset đã có đủ train/val/test và hai lớp NORMAL, PNEUMONIA."""
    data_dir = Path(data_dir)
    if not data_dir.exists():
        return False

    for split in ('train', 'val', 'test'):
        for class_name in CLASSES:
            folder = data_dir / split / class_name
            if not folder.exists():
                return False

            image_count = sum(
                1 for path in folder.iterdir()
                if path.suffix.lower() in {'.jpeg', '.jpg', '.png', '.bmp', '.webp'}
            )
            if image_count == 0:
                return False

    return True


def find_prepared_dataset():
    """Tự tìm dataset_clean_70_15_15 trên Kaggle/Colab/local."""
    candidates = [
        # Kaggle - trường hợp dataset title giữ nguyên dấu _
        Path('/kaggle/input/dataset_clean/dataset/dataset_clean_70_15_15'),
        Path('/kaggle/input/dataset_clean/dataset_clean_70_15_15'),

        # Kaggle - trường hợp Kaggle đổi dataset_clean thành dataset-clean
        Path('/kaggle/input/dataset-clean/dataset/dataset_clean_70_15_15'),
        Path('/kaggle/input/dataset-clean/dataset_clean_70_15_15'),

        # Một số cấu trúc khác hay gặp
        Path('/kaggle/input/dataset/dataset_clean_70_15_15'),
        Path('/kaggle/input/dataset_clean_70_15_15'),

        # Colab/local cũ
        Path('/content/drive/MyDrive/CNN_ViT/dataset_clean_70_15_15'),
        Path('/content/drive/MyDrive/dataset_clean_70_15_15'),
        Path('/content/dataset_clean_70_15_15'),
        Path.cwd() / 'dataset' / 'dataset_clean_70_15_15',
        Path.cwd() / 'dataset_clean_70_15_15',
    ]

    for candidate in candidates:
        if prepared_dataset_is_complete(candidate):
            return candidate

    # Quét rộng trong /kaggle/input để tránh sai tên slug dataset.
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        for candidate in kaggle_input.rglob(DATASET_FOLDER_NAME):
            if prepared_dataset_is_complete(candidate):
                return candidate

    raise FileNotFoundError(
        'Không tìm thấy dataset hợp lệ. Hãy kiểm tra bạn đã Add Input trên Kaggle chưa.\n'
        'Cấu trúc cần có dạng:\n'
        '/kaggle/input/<ten-dataset>/dataset/dataset_clean_70_15_15/train/NORMAL\n'
        '/kaggle/input/<ten-dataset>/dataset/dataset_clean_70_15_15/train/PNEUMONIA\n'
        'và tương tự cho val/test.'
    )


PREPARED_DATA_DIR = find_prepared_dataset()

# Kaggle không cho ghi vào /kaggle/input, nên lưu model/ảnh kết quả vào /kaggle/working.
BASE_OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else PREPARED_DATA_DIR.parent

print(f'Sử dụng dataset tại: {PREPARED_DATA_DIR.resolve()}')
print(f'Lưu kết quả tại: {BASE_OUTPUT_DIR.resolve()}')

# Hỗ trợ cả split_summary.json và split_sumary.json nếu file bị đặt tên thiếu chữ m.
for summary_name in ('split_summary.json', 'split_sumary.json'):
    summary_path = PREPARED_DATA_DIR / summary_name
    if summary_path.exists():
        with open(summary_path, 'r', encoding='utf-8') as file:
            print(json.dumps(json.load(file), indent=4, ensure_ascii=False))
        break


In [ ]:
from pathlib import Path
import copy
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

SEED = 42
DATA_DIR = PREPARED_DATA_DIR
OUTPUT_DIR = BASE_OUTPUT_DIR / 'outputs' / 'vit_b_16'
IMAGE_SIZE = 224
PREPROCESS_VERSION = 'center_crop_v1'
BATCH_SIZE = 32  # Dung T4 x2 thi 32 kha on; neu het VRAM thi giam ve 16 hoac 8.
EPOCHS = 25
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0  # Doi thanh 2 trong Colab neu muon nap du lieu nhanh hon.
PATIENCE = 3
FREEZE_BACKBONE = False  # Dat True neu muon chay thu nhanh hon.

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f'Device: {DEVICE}')
print(f'GPU count: {NUM_GPUS}')
if NUM_GPUS > 0:
    for gpu_idx in range(NUM_GPUS):
        print(f'GPU {gpu_idx}: {torch.cuda.get_device_name(gpu_idx)}')
print(f'Dataset path: {DATA_DIR.resolve()}')
if not DATA_DIR.exists():
    raise FileNotFoundError(f'Khong tim thay dataset tai: {DATA_DIR.resolve()}')

## 2. Doc du lieu, chuan hoa anh va xem phan bo lop

Anh goc khong can chuan hoa truoc tren o dia. Pipeline duoi day tu dong doi anh X-quang thanh 3 kenh, resize theo kich thuoc dau vao model, chuyen gia tri pixel ve tensor va normalize theo ImageNet vi model su dung pretrained weights. Viec nay dien ra khi doc batch va khong lam thay doi tep anh goc.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize(IMAGE_SIZE, antialias=True),
    transforms.CenterCrop((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(7),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize(IMAGE_SIZE, antialias=True),
    transforms.CenterCrop((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = datasets.ImageFolder(DATA_DIR / 'train', transform=train_transform)
val_dataset = datasets.ImageFolder(DATA_DIR / 'val', transform=eval_transform)
test_dataset = datasets.ImageFolder(DATA_DIR / 'test', transform=eval_transform)
class_names = train_dataset.classes
num_classes = len(class_names)

for split_name, split_dataset in [('train', train_dataset), ('val', val_dataset), ('test', test_dataset)]:
    counts = np.bincount(split_dataset.targets, minlength=num_classes)
    print(f'{split_name:5s}: ' + ', '.join(f'{name}={count}' for name, count in zip(class_names, counts)))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (split_name, split_dataset) in zip(axes, [('train', train_dataset), ('val', val_dataset), ('test', test_dataset)]):
    counts = np.bincount(split_dataset.targets, minlength=num_classes)
    ax.bar(class_names, counts, color=['#dadaeb', '#756bb1'])
    ax.set_title(split_name)
    ax.set_ylabel('So anh')
plt.suptitle('Phan bo du lieu theo lop')
plt.tight_layout()
plt.show()

pin_memory = DEVICE.type == 'cuda'
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=pin_memory)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_memory)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_memory)

## 3. Khoi tao ViT-B/16 va cac ham huan luyen

In [ ]:
model = models.vit_b_16(weights=None)
if FREEZE_BACKBONE:
    for parameter in model.parameters():
        parameter.requires_grad = False
model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

# =========================
# DUNG 2 GPU T4 x2 NEU KAGGLE CAP 2 GPU
# =========================
model = model.to(DEVICE)
if DEVICE.type == 'cuda' and torch.cuda.device_count() > 1:
    print(f'Dang dung DataParallel voi {torch.cuda.device_count()} GPU')
    model = nn.DataParallel(model)
else:
    print('Dang dung 1 GPU hoac CPU')

def get_raw_model(model):
    """Lay model goc khi co boc nn.DataParallel."""
    return model.module if isinstance(model, nn.DataParallel) else model

def get_clean_state_dict(model):
    """Luu state_dict khong kem tien to module., de load lai de hon."""
    raw_model = get_raw_model(model)
    return {key: value.detach().cpu().clone() for key, value in raw_model.state_dict().items()}

def load_clean_state_dict(model, state_dict):
    raw_model = get_raw_model(model)
    raw_model.load_state_dict(state_dict)

positive_class_idx = class_names.index('PNEUMONIA')
train_counts = np.bincount(train_dataset.targets, minlength=num_classes)
class_weights = len(train_dataset) / (num_classes * train_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

print(f'Classes: {class_names}')
print(f'Class weights: {class_weights.detach().cpu().numpy()}')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

def run_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None
    model.train(is_training)
    running_loss = 0.0
    labels_all, probs_all, predictions_all = [], [], []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = model(images)
                loss = criterion(logits, labels)
            if is_training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        running_loss += loss.item() * images.size(0)
        probabilities = torch.softmax(logits, dim=1)
        positive_probabilities = probabilities[:, positive_class_idx]
        predictions = probabilities.argmax(dim=1)
        labels_all.extend(labels.detach().cpu().numpy())
        probs_all.extend(positive_probabilities.detach().cpu().numpy())
        predictions_all.extend(predictions.detach().cpu().numpy())

    labels_all = np.asarray(labels_all)
    probs_all = np.asarray(probs_all)
    predictions = np.asarray(predictions_all)
    return {
        'loss': running_loss / len(loader.dataset),
        'accuracy': accuracy_score(labels_all, predictions),
        'auc': roc_auc_score(labels_all, probs_all),
        'labels': labels_all,
        'probs': probs_all,
        'predictions': predictions,
    }

## 4. Huan luyen va luu checkpoint tot nhat

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_accuracy': [], 'val_accuracy': [], 'train_auc': [], 'val_auc': []}
best_val_auc = -1.0
best_state = None
epochs_without_improvement = 0
checkpoint_path = OUTPUT_DIR / 'best_vit_b_16_2gpu.pth'
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, criterion, optimizer)
    val_metrics = run_epoch(model, val_loader, criterion)
    scheduler.step(val_metrics['auc'])
    for metric in ['loss', 'accuracy', 'auc']:
        history[f'train_{metric}'].append(train_metrics[metric])
        history[f'val_{metric}'].append(val_metrics[metric])

    print(f"Epoch {epoch:02d}/{EPOCHS} | train loss={train_metrics['loss']:.4f} acc={train_metrics['accuracy']:.4f} auc={train_metrics['auc']:.4f} | val loss={val_metrics['loss']:.4f} acc={val_metrics['accuracy']:.4f} auc={val_metrics['auc']:.4f}")
    if val_metrics['auc'] > best_val_auc:
        best_val_auc = val_metrics['auc']
        best_state = get_clean_state_dict(model)
        torch.save({'model_state_dict': best_state, 'class_names': class_names, 'val_auc': float(best_val_auc), 'preprocess_version': PREPROCESS_VERSION, 'data_dir': DATA_DIR.name, 'multi_gpu': torch.cuda.device_count() > 1}, checkpoint_path)
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print('Early stopping: validation AUC khong cai thien.')
            break

elapsed_minutes = (time.time() - start_time) / 60
if best_state is None:
    raise RuntimeError('Chưa có checkpoint tốt nhất. Hãy kiểm tra lại quá trình train/validation.')
load_clean_state_dict(model, best_state)
print(f'Hoan tat sau {elapsed_minutes:.1f} phut. Best val AUC: {best_val_auc:.4f}')
print(f'Checkpoint: {checkpoint_path}')

## 5. Bieu do qua trinh huan luyen

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, metric, title in zip(axes, ['loss', 'accuracy', 'auc'], ['Loss', 'Accuracy', 'ROC AUC']):
    ax.plot(epochs_ran, history[f'train_{metric}'], marker='o', label='Train')
    ax.plot(epochs_ran, history[f'val_{metric}'], marker='o', label='Validation')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle('ViT-B/16 training history', fontsize=14)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'training_curves.png', dpi=200, bbox_inches='tight')
plt.show()

## 6. Danh gia tren tap test: report, confusion matrix va ROC

In [ ]:
test_metrics = run_epoch(model, test_loader, criterion)
print(f"Test loss: {test_metrics['loss']:.4f} | Test accuracy: {test_metrics['accuracy']:.4f} | Test AUC: {test_metrics['auc']:.4f}")
print('\nClassification report:')
print(classification_report(test_metrics['labels'], test_metrics['predictions'], target_names=class_names, digits=4))

cm = confusion_matrix(test_metrics['labels'], test_metrics['predictions'])
fpr, tpr, _ = roc_curve(test_metrics['labels'], test_metrics['probs'])
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
image = axes[0].imshow(cm, interpolation='nearest', cmap='Purples')
fig.colorbar(image, ax=axes[0], fraction=0.046, pad=0.04)
axes[0].set_xticks(range(len(class_names)), labels=class_names)
axes[0].set_yticks(range(len(class_names)), labels=class_names)
threshold = cm.max() / 2
for row in range(cm.shape[0]):
    for col in range(cm.shape[1]):
        axes[0].text(col, row, cm[row, col], ha='center', va='center', color='white' if cm[row, col] > threshold else 'black')
axes[0].set_title('Confusion matrix - Test')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[1].plot(fpr, tpr, color='purple', label=f"AUC = {test_metrics['auc']:.4f}")
axes[1].plot([0, 1], [0, 1], '--', color='gray')
axes[1].set_title('ROC curve - Test')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'test_evaluation.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. Hien thi mot so du doan

In [ ]:
def denormalize(image):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (image.cpu() * std + mean).clamp(0, 1)

images, labels = next(iter(test_loader))
with torch.no_grad():
    with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
        probabilities = torch.softmax(model(images.to(DEVICE)), dim=1).cpu()
predictions = probabilities.argmax(dim=1)
show_count = min(8, len(images))
fig, axes = plt.subplots(2, 4, figsize=(13, 7))
for idx, ax in enumerate(axes.flatten()):
    if idx >= show_count:
        ax.axis('off')
        continue
    ax.imshow(denormalize(images[idx]).permute(1, 2, 0))
    actual = class_names[labels[idx]]
    predicted = class_names[predictions[idx]]
    confidence = probabilities[idx, predictions[idx]].item()
    color = 'green' if labels[idx] == predictions[idx] else 'red'
    ax.set_title(f'That: {actual}\nDoan: {predicted} ({confidence:.1%})', color=color, fontsize=9)
    ax.axis('off')
plt.suptitle('ViT-B/16 - Du doan mau tren test')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'sample_predictions.png', dpi=200, bbox_inches='tight')
plt.show()

## 8. Kiem tra file output va tai model

Cell nay in toan bo file trong `/kaggle/working/outputs/vit_b_16` va tao link tai checkpoint `.pth`.


In [ ]:
from pathlib import Path

print('Thu muc output:', OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file():
        print(path)

try:
    from IPython.display import FileLink, display
    display(FileLink(str(checkpoint_path)))
except Exception as error:
    print('Khong tao duoc FileLink:', error)

print('Luu y: Neu muon Kaggle giu file sau khi tat session, bam Save Version -> Save & Run All.')
